# VetPivot Career Agent Demo

VetPivot Career Agent helps transitioning service members and veterans translate military experience into civilian resume language, evaluate job fit against a target role, and identify gaps or safety risks before applying. This notebook is an offline, deterministic walkthrough for Kaggle capstone judges.

## Problem

Military experience is often difficult to communicate in civilian hiring language. Valuable leadership, operations, maintenance, safety, logistics, and accountability work can be hidden behind military terminology. AI-generated career advice also creates risk if it invents credentials, changes important facts, or overstates job fit.

## Track: Agents for Good

This project fits the Agents for Good track because it supports veterans during the military-to-civilian career transition while adding safety and evaluation checks to avoid misleading resume claims.

## Architecture

```mermaid
flowchart TD
    A[User input] --> B[Orchestrator]
    B --> C[Resume Agent]
    C --> D[Job Fit Agent]
    D --> E[Evaluation Agent]
    E --> F[Structured JSON report]
```

The notebook uses deterministic mock mode. No live backend, Google credentials, database, or frontend is required.

## Agent Roles

- **Resume Agent:** translates military experience into civilian resume language and ATS-aligned wording.
- **Job Fit Agent:** compares the experience against the target job description and identifies matched/missing keywords.
- **Evaluation Agent:** checks unsupported claims, factual drift, and safety risks.

## Tool Usage

The project includes a VetPivot backend translation tool for backend-enabled modes. This notebook intentionally runs in `mock` mode so judging is offline and reproducible.

## Google ADK Alignment

The project includes an ADK-facing `root_agent`, specialized sub-agent definitions, and a function-tool wrapper. The local notebook uses the same orchestrator workflow in deterministic mock mode.

## Safety / Evaluation

Safety checks focus on unsupported credentials, degrees, changed dollar amounts, changed team size, changed years of experience, job-title overclaims, and overstated fit. The safety-risk demo intentionally includes unsupported senior-role requirements.

## Known Limitations

- Mock mode is deterministic and not a full LLM resume writer.
- Live backend and Gemini/ADK execution are optional and not required here.
- No frontend, database, auth, deployment, job tracking, upload parsing, long-term memory, or MOS database is included.

In [20]:
from pathlib import Path
import json
import sys

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'vetpivot').exists() and (candidate / 'examples').exists():
            return candidate
    fallback = (current / '..').resolve()
    if (fallback / 'src' / 'vetpivot').exists():
        return fallback
    raise RuntimeError('Could not find repository root')

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from vetpivot.orchestrator import run_workflow
from vetpivot.schemas import MissionInput

repo_root


PosixPath('/Users/charlesd/Developer/active/vetpivot-career-agent')

In [17]:
case_paths = {
    'strong_match': repo_root / 'examples' / 'strong_match.json',
    'partial_match': repo_root / 'examples' / 'partial_match.json',
    'safety_risk_overclaim': repo_root / 'examples' / 'safety_risk_overclaim.json',
}

def load_case(path: Path) -> MissionInput:
    payload = json.loads(path.read_text())
    return MissionInput(
        military_experience=payload['military_experience'],
        mos_branch=payload.get('mos_branch', ''),
        target_job_description=payload['target_job_description'],
    )

reports = {name: run_workflow(load_case(path), mode='mock') for name, path in case_paths.items()}
list(reports)


['strong_match', 'partial_match', 'safety_risk_overclaim']

In [21]:
def compact_summary(report):
    return {
        'mode': report.mode,
        'fit_label': report.job_fit.fit_label,
        'missing_keywords': report.job_fit.missing_keywords,
        'safety_flags': report.evaluation.safety_flags,
        'unsupported_claims': report.evaluation.unsupported_claims,
    }

for name, report in reports.items():
    print(f'\n=== {name} summary ===')
    print(json.dumps(compact_summary(report), indent=2))



=== strong_match summary ===
{
  "mode": "mock",
  "fit_label": "Strong Match",
  "missing_keywords": [
    "compliance",
    "inventory"
  ],
  "safety_flags": [
    "No obvious unsupported claims or factual drift detected in the deterministic review."
  ],
  "unsupported_claims": []
}

=== partial_match summary ===
{
  "mode": "mock",
  "fit_label": "Partial Match",
  "missing_keywords": [
    "communication",
    "project"
  ],
  "safety_flags": [
    "No obvious unsupported claims or factual drift detected in the deterministic review."
  ],
  "unsupported_claims": []
}

=== safety_risk_overclaim summary ===
{
  "mode": "mock",
  "fit_label": "Partial Match",
  "missing_keywords": [
    "compliance"
  ],
  "safety_flags": [
    "Target role includes high-risk requirements not supported by the original experience."
  ],
  "unsupported_claims": [
    "target mentions bachelor not supported by source",
    "target mentions certification not supported by source",
    "target mentions d

In [22]:
for name, report in reports.items():
    print(f'\n=== {name} full structured output ===')
    print(json.dumps(report.to_dict(), indent=2))



=== strong_match full structured output ===
{
  "input": {
    "military_experience": "Led a team of 12 soldiers responsible for maintaining mission-critical communications equipment valued at $2.3M while coordinating preventive maintenance schedules and safety checks.",
    "target_job_description": "Operations coordinator responsible for team coordination, equipment inventory, maintenance scheduling, safety compliance, communication with stakeholders, and process improvement across daily operations.",
    "mos_branch": "Army communications team leader"
  },
  "resume": {
    "professional_resume_bullet": "Translated military experience (Army communications team leader): Led a team of 12 soldiers responsible for maintaining mission-critical communications equipment valued at $2.3M while coordinating preventive maintenance schedules and safety checks, emphasizing leadership, operational coordination, accountability, and measurable mission support in civilian terms.",
    "ats_optimize